# Predictor before: preserve mass, then Top-32 hard-prune
Test only. Use an existing predictor checkpoint (5 or 10 epochs). No teacher targets or training.
Two GPUs, FP32, 64 samples/GPU, greedy. Stress on both GPUs must pass before full test.


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
SOURCE=Path('/kaggle/working/Log_Metrics')
BRANCH='smolvlm-attention'
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/huyvanzzz/Log_Metrics.git',str(SOURCE)],check=True)
else:
    branch=subprocess.check_output(['git','-C',str(SOURCE),'branch','--show-current'],text=True).strip()
    if branch != BRANCH: raise RuntimeError('Wrong source branch')
    dirty=subprocess.check_output(['git','-C',str(SOURCE),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if dirty: raise RuntimeError('Tracked local edits detected; no files overwritten')
    subprocess.run(['git','-C',str(SOURCE),'pull','--ff-only','origin',BRANCH],check=True)
print('Source commit:',subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-r',str(SOURCE/'smolvlm_map_predictor/requirements.txt')],check=True)
sys.path.insert(0,str(SOURCE))
os.chdir(SOURCE)


In [ ]:
VARIANT='before'
PREDICTOR_CHECKPOINT='' # /kaggle/input/.../predictor_best.pt
PROJECTED_CACHE_INPUT='' # directory containing projected.npy/index.json/manifest.json
PROJECTED_CACHE_DIR='/kaggle/working/smolvlm_projected_visual_cache'
CACHE_INPUT='' # raw feature cache only if building projected cache
PREPARED_DIR='' # blank downloads pinned public prepared data
BASE_CHECKPOINT='' # blank downloads pinned public base best
OUTPUT=Path('/kaggle/working/smolvlm_predictor_before_preserve_mass_outputs')
RESUME=False
CONFIG=SOURCE/'configs/smolvlm_predictor_before_preserve_mass.json'
assert PREDICTOR_CHECKPOINT and Path(PREDICTOR_CHECKPOINT).is_file(), 'Set PREDICTOR_CHECKPOINT'
from smolvlm_predictor_io.progress import run_notebook
def run(command,*args):
    return run_notebook('smolvlm_predictor_mass',command,args,OUTPUT,SOURCE)


In [ ]:
args=['--config',str(CONFIG),'--output',str(OUTPUT),'--predictor',PREDICTOR_CHECKPOINT,
      '--projected-cache-input',PROJECTED_CACHE_INPUT,'--projected-cache-dir',PROJECTED_CACHE_DIR,'--cache-input',CACHE_INPUT]
if PREPARED_DIR: args += ['--prepared-dir',PREPARED_DIR]
if BASE_CHECKPOINT: args += ['--base-checkpoint',BASE_CHECKPOINT]
if RESUME: args += ['--resume']
run('prepare',*args)


In [ ]:
run('evaluate','--run',str(OUTPUT/'run.json'))


In [ ]:
run('bundle','--run',str(OUTPUT/'run.json'))
print(OUTPUT/'predictor_preserve_mass_artifacts.zip')
